In [1]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Load cleaned datasets
layoffs = pd.read_csv('../data/cleaned/layoffs_clean.csv')
placement = pd.read_csv('../data/cleaned/placement_clean.csv')
hr = pd.read_csv('../data/cleaned/hr_clean.csv')

# Create SQLite database in memory
conn = sqlite3.connect(':memory:')

# Load dataframes into SQL tables
layoffs.to_sql('layoffs', conn, index=False, if_exists='replace')
placement.to_sql('placement', conn, index=False, if_exists='replace')
hr.to_sql('hr', conn, index=False, if_exists='replace')

print("✅ SQL Database created successfully!")
print("📊 Tables created: layoffs, placement, hr")

✅ SQL Database created successfully!
📊 Tables created: layoffs, placement, hr


In [2]:
query1 = """
SELECT 
    company,
    SUM(total_laid_off) as total_layoffs,
    COUNT(*) as layoff_rounds
FROM layoffs
WHERE total_laid_off > 0
GROUP BY company
ORDER BY total_layoffs DESC
LIMIT 10;
"""

result1 = pd.read_sql_query(query1, conn)
print("🔴 Query 1: Top 10 Companies by Total Layoffs")
print("=" * 50)
print(result1.to_string(index=False))

🔴 Query 1: Top 10 Companies by Total Layoffs
   company  total_layoffs  layoff_rounds
    Amazon        58124.0             13
     Intel        43115.0              9
      Meta        35700.0              9
    Oracle        31294.0              8
 Microsoft        30055.0             10
      Dell        23650.0              3
     Cisco        18521.0              6
Salesforce        16525.0             12
     Tesla        14500.0              2
    Google        13697.0             10


In [3]:
query2 = """
SELECT 
    industry,
    SUM(total_laid_off) as total_layoffs,
    COUNT(DISTINCT company) as companies_affected,
    ROUND(AVG(total_laid_off), 0) as avg_layoffs_per_round
FROM layoffs
WHERE total_laid_off > 0
AND industry != 'Unknown'
GROUP BY industry
ORDER BY total_layoffs DESC;
"""

result2 = pd.read_sql_query(query2, conn)
print("🔴 Query 2: Industry Wise Layoff Analysis")
print("=" * 50)
print(result2.to_string(index=False))

🔴 Query 2: Industry Wise Layoff Analysis
      industry  total_layoffs  companies_affected  avg_layoffs_per_round
         Other       119746.0                 123                  634.0
        Retail       106706.0                 157                  450.0
      Hardware       106157.0                  32                 1930.0
      Consumer        96932.0                 120                  490.0
       Finance        67565.0                 253                  194.0
Transportation        66002.0                 112                  361.0
          Food        52091.0                 104                  324.0
    Healthcare        39244.0                 164                  190.0
Infrastructure        24835.0                  29                  606.0
        Travel        23740.0                  49                  317.0
         Sales        21905.0                  35                  365.0
     Education        21002.0                  72                  193.0
   Real Es

In [4]:
query3 = """
SELECT 
    country,
    SUM(total_laid_off) as total_layoffs,
    COUNT(DISTINCT company) as companies_affected
FROM layoffs
WHERE total_laid_off > 0
AND country != 'Unknown'
GROUP BY country
ORDER BY total_layoffs DESC
LIMIT 10;
"""

result3 = pd.read_sql_query(query3, conn)
print("🔴 Query 3: Top 10 Countries by Layoffs")
print("=" * 50)
print(result3.to_string(index=False))

🔴 Query 3: Top 10 Countries by Layoffs
       country  total_layoffs  companies_affected
 United States       656271.0                1189
         India        66039.0                 181
       Germany        32055.0                  74
United Kingdom        23354.0                  70
   Netherlands        21575.0                  13
        Sweden        20379.0                  20
        Canada        16068.0                  97
        Israel        14309.0                  95
        Brazil        11939.0                  58
         China         8190.0                  11


In [5]:
query4 = """
SELECT 
    college_tier,
    COUNT(*) as total_students,
    SUM(CASE WHEN placement_status = 1 THEN 1 ELSE 0 END) as placed,
    SUM(CASE WHEN placement_status = 0 THEN 1 ELSE 0 END) as not_placed,
    ROUND(100.0 * SUM(CASE WHEN placement_status = 1 THEN 1 ELSE 0 END) 
          / COUNT(*), 2) as placement_rate_percent
FROM placement
GROUP BY college_tier
ORDER BY placement_rate_percent DESC;
"""

result4 = pd.read_sql_query(query4, conn)
print("🟡 Query 4: Placement Rate by College Tier")
print("=" * 50)
print(result4.to_string(index=False))

🟡 Query 4: Placement Rate by College Tier
college_tier  total_students  placed  not_placed  placement_rate_percent
      Tier-1           14868   12349        2519                   83.06
      Tier-2           39910   28959       10951                   72.56
      Tier-3           45222   27167       18055                   60.07


In [6]:
query5 = """
SELECT 
    branch,
    COUNT(*) as total_placed,
    ROUND(AVG(salary_package_lpa), 2) as avg_salary,
    ROUND(MAX(salary_package_lpa), 2) as max_salary,
    ROUND(MIN(salary_package_lpa), 2) as min_salary
FROM placement
WHERE placement_status = 1
GROUP BY branch
ORDER BY avg_salary DESC
LIMIT 10;
"""

result5 = pd.read_sql_query(query5, conn)
print("🟡 Query 5: Salary Analysis by Branch")
print("=" * 50)
print(result5.to_string(index=False))

🟡 Query 5: Salary Analysis by Branch
  branch  total_placed  avg_salary  max_salary  min_salary
      CE          6482       17.35       28.33        8.54
     CSE         17870       17.33       28.10        7.97
Chemical          6343       17.31       27.64        8.55
      ME          7974       17.30       26.96        6.88
      EE          8023       17.30       26.91        7.56
     ECE         10394       17.29       27.31        8.55
      IT         11389       17.28       27.25        7.39


In [7]:
query6 = """
SELECT 
    CASE 
        WHEN cgpa >= 9.0 THEN '9.0 and above'
        WHEN cgpa >= 8.0 THEN '8.0 to 8.9'
        WHEN cgpa >= 7.0 THEN '7.0 to 7.9'
        WHEN cgpa >= 6.0 THEN '6.0 to 6.9'
        ELSE 'Below 6.0'
    END as cgpa_range,
    COUNT(*) as total_students,
    ROUND(100.0 * SUM(CASE WHEN placement_status = 1 
          THEN 1 ELSE 0 END) / COUNT(*), 2) as placement_rate,
    ROUND(AVG(CASE WHEN placement_status = 1 
          THEN salary_package_lpa END), 2) as avg_salary_if_placed
FROM placement
GROUP BY cgpa_range
ORDER BY placement_rate DESC;
"""

result6 = pd.read_sql_query(query6, conn)
print("🟡 Query 6: CGPA Impact on Placement")
print("=" * 50)
print(result6.to_string(index=False))

🟡 Query 6: CGPA Impact on Placement
   cgpa_range  total_students  placement_rate  avg_salary_if_placed
9.0 and above            2707           83.34                 19.35
   8.0 to 8.9           16962           77.03                 18.36
   7.0 to 7.9           39304           70.81                 17.44
   6.0 to 6.9           31472           63.37                 16.63
    Below 6.0            9555           56.30                 15.71


In [8]:
query7 = """
SELECT 
    Department,
    COUNT(*) as total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) as left_company,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) 
          / COUNT(*), 2) as attrition_rate,
    ROUND(AVG(MonthlyIncome), 0) as avg_monthly_income
FROM hr
GROUP BY Department
ORDER BY attrition_rate DESC;
"""

result7 = pd.read_sql_query(query7, conn)
print("🟢 Query 7: HR Attrition by Department")
print("=" * 50)
print(result7.to_string(index=False))

🟢 Query 7: HR Attrition by Department
            Department  total_employees  left_company  attrition_rate  avg_monthly_income
                 Sales              446            92           20.63              6959.0
       Human Resources               63            12           19.05              6655.0
Research & Development              961           133           13.84              6281.0


In [9]:
query8 = """
SELECT 
    CASE 
        WHEN TotalWorkingYears <= 2 THEN '0-2 years'
        WHEN TotalWorkingYears <= 5 THEN '3-5 years'
        WHEN TotalWorkingYears <= 10 THEN '6-10 years'
        ELSE 'Above 10 years'
    END as experience_range,
    COUNT(*) as employees,
    ROUND(AVG(MonthlyIncome), 0) as avg_salary,
    ROUND(AVG(JobSatisfaction), 2) as avg_job_satisfaction
FROM hr
GROUP BY experience_range
ORDER BY avg_salary DESC;
"""

result8 = pd.read_sql_query(query8, conn)
print("🟢 Query 8: Experience vs Salary Analysis")
print("=" * 50)
print(result8.to_string(index=False))

🟢 Query 8: Experience vs Salary Analysis
experience_range  employees  avg_salary  avg_job_satisfaction
  Above 10 years        547     10020.0                  2.71
      6-10 years        607      5190.0                  2.73
       3-5 years        193      3370.0                  2.78
       0-2 years        123      2259.0                  2.72


In [10]:
query9 = """
SELECT 
    JobRole,
    COUNT(*) as total,
    SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) as left_job,
    ROUND(100.0 * SUM(CASE WHEN Attrition='Yes' 
          THEN 1 ELSE 0 END)/COUNT(*), 2) as attrition_rate,
    ROUND(AVG(MonthlyIncome), 0) as avg_income
FROM hr
GROUP BY JobRole
ORDER BY attrition_rate DESC
LIMIT 5;
"""

result9 = pd.read_sql_query(query9, conn)
print("🟢 Query 9: Top 5 High Risk Job Roles")
print("=" * 50)
print(result9.to_string(index=False))

🟢 Query 9: Top 5 High Risk Job Roles
              JobRole  total  left_job  attrition_rate  avg_income
 Sales Representative     83        33           39.76      2626.0
Laboratory Technician    259        62           23.94      3237.0
      Human Resources     52        12           23.08      4236.0
      Sales Executive    326        57           17.48      6924.0
   Research Scientist    292        47           16.10      3240.0


In [11]:
query10 = """
SELECT 'Total Companies in Layoffs Data' as metric,
        COUNT(DISTINCT company) as value
FROM layoffs

UNION ALL

SELECT 'Total Employees Laid Off',
        SUM(total_laid_off)
FROM layoffs WHERE total_laid_off > 0

UNION ALL

SELECT 'Total Students in Placement Data',
        COUNT(*)
FROM placement

UNION ALL

SELECT 'Total Students Placed',
        SUM(CASE WHEN placement_status=1 THEN 1 ELSE 0 END)
FROM placement

UNION ALL

SELECT 'Total HR Records',
        COUNT(*)
FROM hr

UNION ALL

SELECT 'Employees Who Left Company',
        SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END)
FROM hr;
"""

result10 = pd.read_sql_query(query10, conn)
print("📊 Query 10: Complete Project Summary")
print("=" * 50)
print(result10.to_string(index=False))

# Close connection
conn.close()
print("\n✅ All 10 SQL Queries Complete!")
print("✅ Database connection closed!")

📊 Query 10: Complete Project Summary
                          metric    value
 Total Companies in Layoffs Data   2915.0
        Total Employees Laid Off 920105.0
Total Students in Placement Data 100000.0
           Total Students Placed  68475.0
                Total HR Records   1470.0
      Employees Who Left Company    237.0

✅ All 10 SQL Queries Complete!
✅ Database connection closed!


In [12]:
# Reopen connection to save results
conn = sqlite3.connect(':memory:')
layoffs.to_sql('layoffs', conn, index=False, if_exists='replace')
placement.to_sql('placement', conn, index=False, if_exists='replace')
hr.to_sql('hr', conn, index=False, if_exists='replace')

# Save all query results to one Excel file
with pd.ExcelWriter('../sql/sql_analysis_results.xlsx') as writer:
    result1.to_excel(writer, sheet_name='Top10_Companies', index=False)
    result2.to_excel(writer, sheet_name='Industry_Analysis', index=False)
    result3.to_excel(writer, sheet_name='Country_Layoffs', index=False)
    result4.to_excel(writer, sheet_name='Placement_by_Tier', index=False)
    result5.to_excel(writer, sheet_name='Salary_by_Branch', index=False)
    result6.to_excel(writer, sheet_name='CGPA_Impact', index=False)
    result7.to_excel(writer, sheet_name='HR_Attrition', index=False)
    result8.to_excel(writer, sheet_name='Experience_Salary', index=False)
    result9.to_excel(writer, sheet_name='High_Risk_Roles', index=False)
    result10.to_excel(writer, sheet_name='Project_Summary', index=False)

conn.close()
print("✅ All SQL results saved to Excel!")
print("📁 Location: sql/sql_analysis_results.xlsx")

✅ All SQL results saved to Excel!
📁 Location: sql/sql_analysis_results.xlsx
